In [1]:
!pip install datasets pandas sacrebleu transformers torch sentencepiece scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sacrebleu]


In [2]:
import re, random
import numpy as np, pandas as pd, torch
from scipy.stats import kendalltau
from sacrebleu.metrics import BLEU, CHRF
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset

In [3]:
ds = load_dataset("Nicolas-BZRD/Parallel_Global_Voices_English_French", split="train")
df = ds.to_pandas()[["en", "fr"]]
df["row_id"] = df.index
print("Full corpus:", df.shape)

SEED, N = 42, 1000
words = df["en"].str.split().str.len()
clean = df[df.en.str.strip().ne("") & df.fr.str.strip().ne("") & words.between(5, 80)]
sample = clean.sample(N, random_state=SEED).sort_values("row_id").reset_index(drop=True)

print(f"Seed: {SEED} | sample size: {len(sample)} (drawn from {len(clean)} pairs with 5-80 English words)")
print("Sample row IDs:")
print(sample.row_id.tolist())

README.md:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

data/train-00000-of-00001-39372d4d4f86fc(…): reconstructing file:   0%|          |  0.00B / 57.7MB            

data/train-00000-of-00001-39372d4d4f86fc(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/342060 [00:00<?, ? examples/s]

Full corpus: (342060, 3)
Seed: 42 | sample size: 1000 (drawn from 327393 pairs with 5-80 English words)
Sample row IDs:
[135, 315, 445, 683, 800, 912, 1399, 1452, 1685, 2108, 2318, 2991, 3535, 4292, 5200, 5268, 5333, 6061, 6084, 6127, 6354, 6506, 6656, 7199, 7293, 7493, 7604, 8226, 9250, 9721, 10354, 10604, 10698, 10939, 11023, 11670, 12842, 13391, 14843, 15458, 15933, 16110, 16218, 16289, 16598, 16653, 17628, 19070, 19580, 19877, 20157, 20883, 23484, 24882, 25260, 25599, 25714, 26425, 26704, 26765, 26812, 26965, 27239, 27613, 27740, 27790, 28232, 28316, 28342, 28403, 28488, 28801, 29581, 30347, 30654, 30859, 31302, 31534, 31609, 32009, 32222, 33268, 33278, 33311, 34109, 34349, 35102, 35466, 35623, 35691, 36311, 36760, 37089, 37180, 38078, 38774, 39061, 39453, 39460, 39529, 39632, 39837, 40042, 40294, 40659, 40725, 40744, 41758, 41817, 42056, 42370, 42661, 42665, 42840, 43476, 44033, 44034, 44326, 44695, 45189, 45448, 45556, 46609, 47386, 47502, 47577, 47786, 47964, 48567, 48717, 49069

In [4]:
MODEL = "Helsinki-NLP/opus-mt-en-fr"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = MarianTokenizer.from_pretrained(MODEL)
model = MarianMTModel.from_pretrained(MODEL).to(device).eval()

def translate(texts, batch_size=32):
    order = np.argsort([len(t) for t in texts])
    out = [None] * len(texts)
    for i in range(0, len(texts), batch_size):
        ids = order[i:i + batch_size]
        batch = tok([texts[j] for j in ids], return_tensors="pt", padding=True,
                    truncation=True, max_length=512).to(device)
        with torch.no_grad():
            gen = model.generate(**batch, num_beams=4, max_new_tokens=256)
        for j, h in zip(ids, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = h
    return out

sample["hyp"] = translate(sample["en"].tolist())
sample[["row_id", "en", "fr", "hyp"]].head(5)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,row_id,en,fr,hyp
0,135,On top of defending themselves from racism….th...,"The way the discussion went, if you asked a pe...",En plus de se défendre contre le racisme... il...
1,315,The super shield protects from extremist ideas.,[…] Le Super Bouclier protège des idées extrém...,Le super bouclier protège des idées extrémistes.
2,445,Strike orders issued by the Madhesi People's R...,Les ordres de grève publiés par le Forum des D...,Les ordres de grève émis par le Madhesi People...
3,683,Reaction or lack of reaction from the governme...,La réaction ou le manque de réaction du gouver...,La réaction ou l'absence de réaction du gouver...
4,800,Iranian Women Activists in Pain and Ahmadineja...,Les Femmes Activistes Iraniennes dans la tourm...,Des militantes iraniennes dans la douleur et l...


In [5]:
refs, hyps = sample["fr"].tolist(), sample["hyp"].tolist()
bleu, chrf = BLEU(), CHRF()
b, c = bleu.corpus_score(hyps, [refs]), chrf.corpus_score(hyps, [refs])
print(b); print(c)
print("BLEU signature:", bleu.get_signature())
print("chrF signature:", chrf.get_signature())
sample["chrf"] = [chrf.sentence_score(h, [r]).score for h, r in zip(hyps, refs)]
print("\nSentence-level chrF: mean = %.2f, median = %.2f" % (sample.chrf.mean(), sample.chrf.median()))

BLEU = 34.08 61.6/39.8/27.9/19.7 (BP = 1.000 ratio = 1.000 hyp_len = 24902 ref_len = 24905)
chrF2 = 60.41
BLEU signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
chrF signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0

Sentence-level chrF: mean = 60.77, median = 60.97


In [6]:
def entities(src):
    ents = []
    for m in re.finditer(r"\b[A-Z][\w'’\-]+", src):
        before = src[:m.start()].rstrip()
        if m.start() == 0 or before[-1:] in ".!?:\"“”(":
            continue
        ents.append(m.group())
    return ents

def ne_errors(row):
    return [e for e in entities(row.en) if e.lower() in row.fr.lower() and e.lower() not in row.hyp.lower()]

EN_NEG = re.compile(r"\b(not|no|never|none|nobody|nothing|neither|nor|without|cannot)\b|n['’]t\b", re.I)
FR_NEG = re.compile(r"\bn['’]|\bne\b|\bpas\b|\bjamais\b|\baucun(e|s)?\b|\brien\b|\bpersonne\b|\bsans\b|\bni\b", re.I)

def neg_error(row):
    return bool(FR_NEG.search(row.hyp)) != bool(FR_NEG.search(row.fr))

GLOSSARY = {
    r"\bblogger": r"blogueu|blogeu",
    r"\bnetizen": r"internaute|net-?citoyen",
    r"\bactivist": r"activiste|militant",
    r"\bcensor": r"censur",
    r"\bhuman rights": r"droits (de l['’]homme|humains|de la personne)",
    r"\b(press freedom|freedom of (the press|expression|speech))": r"libert[ée] (de la presse|d['’]expression|de parole)",
    r"\bgovernment": r"gouvernement",
    r"\bprotest": r"manifest|protest",
    r"\belection": r"[ée]lection|scrutin",
    r"\bcitizen media": r"m[ée]dias? citoyen",
    r"\bsocial (media|networks?)": r"r[ée]seaux? sociaux|m[ée]dias? sociaux",
    r"\brefugee": r"r[ée]fugi",
    r"\bindigenous": r"autochtone|indig[èe]ne",
    r"\bcrackdown": r"r[ée]pression|r[ée]primer",
}
def term_errors(row):
    return [en for en, fr in GLOSSARY.items()
            if re.search(en, row.en, re.I) and re.search(fr, row.fr, re.I) and not re.search(fr, row.hyp, re.I)]

def wtoks(t): return re.findall(r"\w+", t.lower())
def order_tau(row):
    h, r = wtoks(row.hyp), wtoks(row.fr)
    hp, rp = {}, {}
    for i, w in enumerate(h): hp.setdefault(w, i)
    for i, w in enumerate(r): rp.setdefault(w, i)
    shared = [w for w in hp if w in rp]
    if len(shared) < 6: return np.nan
    return kendalltau([hp[w] for w in shared], [rp[w] for w in shared])[0]

def digits(t): return {re.sub(r"\D", "", x) for x in re.findall(r"\d[\d.,\s]*\d|\d", t)}
def meaning_reasons(row):
    reasons = []
    if digits(row.en) - digits(row.hyp): reasons.append("number missing/changed")
    ratio = len(wtoks(row.hyp)) / max(1, len(wtoks(row.fr)))
    if ratio < 0.6: reasons.append(f"possible omission (len ratio {ratio:.2f})")
    if ratio > 1.6: reasons.append(f"possible addition (len ratio {ratio:.2f})")
    if row.chrf < 30: reasons.append(f"very low chrF ({row.chrf:.1f})")
    return reasons

sample["ne_err"] = sample.apply(ne_errors, axis=1)
sample["neg_err"] = sample.apply(neg_error, axis=1)
sample["term_err"] = sample.apply(term_errors, axis=1)
sample["tau"] = sample.apply(order_tau, axis=1)
sample["meaning_err"] = sample.apply(meaning_reasons, axis=1)

flags = {
    "named entities": sample.ne_err.str.len() > 0,
    "negation": sample.neg_err,
    "terminology": sample.term_err.str.len() > 0,
    "word order": sample.tau < 0.5,
    "meaning preservation": sample.meaning_err.str.len() > 0,
}
summary = pd.DataFrame({"flagged": {k: int(v.sum()) for k, v in flags.items()}})
summary["% of sample"] = (100 * summary.flagged / len(sample)).round(1)
summary["mean chrF (flagged)"] = [round(sample.chrf[v].mean(), 1) for v in flags.values()]
print("Mean chrF (all):", round(sample.chrf.mean(), 1))
summary

Mean chrF (all): 60.8


,flagged,% of sample,mean chrF (flagged)
named entities,42,4.2,58.0
negation,69,6.9,51.3
terminology,7,0.7,60.0
word order,51,5.1,49.5
meaning preservation,74,7.4,36.8


In [7]:
detail = {
    "named entities": lambda r: f"entities lost/altered: {r.ne_err}",
    "negation": lambda r: f"negation in REF: {bool(FR_NEG.search(r.fr))} | in HYP: {bool(FR_NEG.search(r.hyp))}",
    "terminology": lambda r: f"glossary terms not matched: {r.term_err}",
    "word order": lambda r: f"Kendall tau (shared words): {r.tau:.2f}",
    "meaning preservation": lambda r: f"reasons: {r.meaning_err}",
}
K = 5
for cat, mask in flags.items():
    sub = sample[mask]
    print("=" * 100); print(f"{cat.upper()}  ({len(sub)} flagged) - {min(K, len(sub))} random examples (seed {SEED})")
    for _, r in sub.sample(min(K, len(sub)), random_state=SEED).iterrows():
        print(f"\n[row {r.row_id}] {detail[cat](r)}\n  SRC: {r.en}\n  REF: {r.fr}\n  HYP: {r.hyp}")

print("=" * 100); print(f"LOWEST sentence-level chrF - 5 worst, for manual inspection")
for _, r in sample.nsmallest(5, "chrf").iterrows():
    print(f"\n[row {r.row_id}] chrF={r.chrf:.1f}\n  SRC: {r.en}\n  REF: {r.fr}\n  HYP: {r.hyp}")

NAMED ENTITIES  (42 flagged) - 5 random examples (seed 42)

[row 213871] entities lost/altered: ['Ghana']
  SRC: The atmosphere in Parliament, televised by Ghana Television (GTV), was that of shock and sadness as Members of Parliament displayed deeply saddened faces.
  REF: L'atmosphère au Parlement, retransmise par la télévision publique du Ghana (GTV), était celle du choc et de la tristesse profonde affichée sur les visages des députés.
  HYP: L'atmosphère au Parlement, diffusée par la télévision ghanéenne (GTV), a été celle du choc et de la tristesse, alors que les députés avaient des visages profondément attristés.

[row 156439] entities lost/altered: ['CT']
  SRC: On Monday morning, March 14, I got a call from a doctor urging me to come to the hospital because a CT Scan was needed.
  REF: Le lundi matin, le 14 mars, j'ai reçu un appel d'un docteur me demandant de venir de toute urgence à l'hôpital car un IRM était nécessaire.
  HYP: Lundi matin, le 14 mars, j'ai reçu un appel d'un

In [9]:
print(f"Corpus BLEU = {b.score:.2f} | corpus chrF = {c.score:.2f}\n")

Corpus BLEU = 34.08 | corpus chrF = 60.41

